# Pharmaceutical Document RAG System

An AI-powered Retrieval-Augmented Generation (RAG) pipeline for analyzing pharmaceutical PDF documents.

**Core workflow:** PDF extraction/OCR → document classification → logical document segmentation → metadata-aware chunking → semantic embeddings → FAISS retrieval → LLM-based query routing → Mistral answer generation → source attribution.

> **Public repository note:** Company-specific test questions, ground-truth answers, and proprietary documents are intentionally excluded from this public version. The notebook retains the technical pipeline while using a public sample PDF for demonstration.

In [ ]:
 # STEP 1: Install Required Packages

# Install required packages
!pip install -q gradio
!pip install -q gradio_pdf
!pip install -q pypdf PyPDF2 pymupdf
!pip install -q sentence-transformers transformers
!pip install -q faiss-cpu
!pip install -q google-generativeai
!pip install -q numpy pandas

# Install LlamaIndex packages for enhanced document processing
!pip install -q llama-index
!pip install -q llama-index-readers-file
!pip install -q llama-index-embeddings-huggingface
!pip install -q llama-index-vector-stores-faiss
!pip install -q llama-index-llms-gemini

In [ ]:
# STEP 2: Core Imports and Configuration

import gradio as gr
from gradio_pdf import PDF
import fitz  # PyMuPDF
from PyPDF2 import PdfReader
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import google.generativeai as genai
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
import json
from datetime import datetime
import hashlib

# LlamaIndex imports for enhanced document processing
from llama_index.core import Document, VectorStoreIndex, StorageContext
from llama_index.core.schema import TextNode
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator



In [ ]:
import torch

# Install packages for efficient model loading
!pip install -q accelerate
!pip install -q bitsandbytes

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Check CUDA version first
!nvcc --version

# Install llama-cpp-python with CUDA 12.x support
!pip install --no-cache-dir llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu123

### Using an Open-Source LLM (Mistral) instead of Gemini

The implementation uses the open-source Mistral 7B Instruct model through `llama_cpp.Llama`.

**Note**: Running this model locally requires sufficient RAM and GPU memory. If you face issues, consider using a smaller model, quantizing the model further, or utilizing a hosted inference endpoint (e.g., from Hugging Face or another provider).

In [ ]:
from llama_cpp import Llama
import os

# Define model path
model_path = "/content/mistral.gguf"

# Download Mistral model if not already present
if not os.path.exists(model_path):
    !wget https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf -O {model_path}
    print(f"Model downloaded to {model_path}")

# Verify model file exists
if os.path.exists(model_path):
    print(f"Model file exists. Size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")
else:
    print("Model file not found!")

# Load the model with GPU acceleration
try:
    llm = Llama(
        model_path=model_path,
        n_gpu_layers=1,  # Start with 1 layer on GPU for efficiency
        n_ctx=2048,      # Context window size
        verbose=True     # Show loading progress
    )

    print("Model loaded successfully!")

except Exception as e:
    print(f"Error loading model: {e}")

In [ ]:
# Initialize embedding models (both for compatibility)
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
llama_embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Imports and configuration complete.")

In [ ]:
# STEP 3: Data Structures for Document Management

@dataclass
class PageInfo:
    """Stores information about a single page"""
    page_num: int
    text: str
    doc_type: Optional[str] = None
    page_in_doc: int = 0

@dataclass
class LogicalDocument:
    """Represents a logical document within a PDF"""
    doc_id: str
    doc_type: str
    page_start: int
    page_end: int
    text: str
    chunks: List[Dict] = None

@dataclass
class ChunkMetadata:
    """Rich metadata for each chunk"""
    chunk_id: str
    doc_id: str
    doc_type: str
    chunk_index: int
    page_start: int
    page_end: int
    text: str
    embedding: Optional[np.ndarray] = None

In [ ]:
VALID_DOC_TYPES = [
    "Cover Letter", "Certificate Of Quality", "Packaging Specification",
    "Bse/Tse Declaration", "Material Description", "Supplier Qualification",
    "Chain Of Custody", "Other"
]

def clean_doc_type(response_text):
    """Clean up LLM response to extract a valid doc_type label."""
    cleaned = response_text.strip().replace('"', '').replace('`', '').replace('*', '').lower().replace(".", "").strip()
    cleaned_title = cleaned.title()
    for label in VALID_DOC_TYPES:
        if label.lower() in cleaned.lower():
            return label
    return cleaned_title


def classify_document_type(text: str, max_length: int = 1500) -> str:
    """
    Classify the document type based on its content.
    Uses LLM to intelligently identify pharmaceutical document category.
    """
    # Truncate text if too long to avoid token limits
    text_sample = text[:max_length] if len(text) > max_length else text

    prompt = f"""You are a pharmaceutical document classifier. Based on the page
content below, classify it into ONE of these document types:

- Cover Letter: A formal letter (often starting with "To Whom It
  May Concern") discussing product information or storage conditions.
- Certificate of Quality: Contains lot numbers, manufacture dates,
  expiration dates, and test results (autoclave, gamma irradiation).
- Packaging Specification: Describes packaging components, materials,
  part numbers, and configuration change history.
- BSE/TSE Declaration: A declaration about animal-origin materials
  and transmissible spongiform encephalopathy compliance.
- Material Description: Lists materials of construction, sterilization
  compatibility, and physical properties of a product.
- Supplier Qualification: Contains supplier audit history,
  certifications (ISO 9001, ISO 13485), and approved product lists.
- Chain of Custody: Lists manufactured assemblies, traceability
  information, and the manufacturing-to-shipment flow.
- Other: Use ONLY if the content does not match any of the above.

Page content:
{text_sample}

Respond with ONLY the document type name. No explanation."""

    try:
        # Call the LLM and get the response text from the choices
        response = llm(prompt, max_tokens=50)
        response_text = response['choices'][0]['text'] # Corrected access
        return clean_doc_type(response_text)
    except Exception as e:
        print(f"Classification error: {e}")
        return 'Other'

def detect_document_boundary(prev_text: str, curr_text: str,
                            current_doc_type: str = None) -> bool:
    """
    Detect if two consecutive pages belong to the same document.
    Returns True if they're from the same document.
    """
    # Quick heuristic checks first
    if not prev_text or not curr_text:
        return False

    # Sample the texts for LLM analysis
    prev_sample = prev_text[-500:] if len(prev_text) > 500 else prev_text
    curr_sample = curr_text[:500] if len(curr_text) > 500 else curr_text

    prompt = f"""Determine if these two pages are from the SAME pharmaceutical document.

Current document type: {current_doc_type or 'Unknown'}

A NEW document starts when the page has:
- A different document title or heading (e.g., "Certificate of Quality"
  vs "Packaging Specification" vs "Material Description Sheet")
- A completely different topic or subject matter
- Its own header with a new document number or reference

Pages belong to the SAME document when:
- The second page says "continued" or "page 2 of 2"
- The content directly continues the previous page's discussion
- They share the same document number or title

End of Previous Page:
...{prev_sample}

Start of Current Page:
{curr_sample}...

Answer ONLY 'Yes' if same document or 'No' if different document."""

    try:
        # Call the LLM and get the response text from the choices
        response = llm(prompt, max_tokens=10)
        response_text = response['choices'][0]['text'] # Corrected access
        return response_text.strip().lower().startswith('yes')
    except Exception as e:
        print(f"Boundary detection error: {e}")
        # Default to keeping pages together if uncertain
        return True

In [ ]:
# STEP 5: Advanced PDF Processing Pipeline

def extract_and_analyze_pdf(pdf_file) -> Tuple[List[PageInfo], List[LogicalDocument]]:
    """
    Extract text from PDF and perform intelligent document analysis.
    Returns both page-level info and logical document groupings.
    Supports various file types including scanned PDFs with OCR.
    """
    print("Starting PDF extraction and analysis...")

    # Extract text from each page
    if isinstance(pdf_file, dict) and "content" in pdf_file:
        doc = fitz.open(stream=pdf_file["content"], filetype="pdf")
    elif hasattr(pdf_file, "read"):
        doc = fitz.open(stream=pdf_file.read(), filetype="pdf")
    else:
        doc = fitz.open(pdf_file)

    pages_info = []
    for i, page in enumerate(doc):
        text = page.get_text()

        # If no text found, try OCR (for scanned documents)
        if not text.strip():
            print(f"  Page {i}: No text found, attempting OCR...")
            try:
                # Convert page to image and perform OCR
                pix = page.get_pixmap()
                img_data = pix.tobytes("png")
                from PIL import Image
                import pytesseract
                import io

                img = Image.open(io.BytesIO(img_data))
                text = pytesseract.image_to_string(img)
                print(f"  Page {i}: OCR extracted {len(text)} characters")
            except Exception as e:
                print(f"  Page {i}: OCR failed - {e}")
                text = ""

        pages_info.append(PageInfo(page_num=i, text=text))

    doc.close()

    if not pages_info:
        raise ValueError("No text could be extracted from PDF")

    print(f"Extracted {len(pages_info)} pages")

    # Perform document classification and boundary detection
    print("Analyzing document structure...")
    logical_docs = []
    current_doc_type = None
    current_doc_pages = []
    doc_counter = 0

    for i, page_info in enumerate(pages_info):
        if i == 0:
            # First page - classify document type
            current_doc_type = classify_document_type(page_info.text)
            page_info.doc_type = current_doc_type
            page_info.page_in_doc = 0
            current_doc_pages = [page_info]
            print(f"  Page {i}: New document detected - {current_doc_type}")
        else:
            # Check if this page continues the previous document
            prev_text = pages_info[i-1].text
            is_same = detect_document_boundary(prev_text, page_info.text, current_doc_type)

            if is_same:
                # Continue current document
                page_info.doc_type = current_doc_type
                page_info.page_in_doc = len(current_doc_pages)
                current_doc_pages.append(page_info)
            else:
                # New document detected - save previous and start new
                logical_doc = LogicalDocument(
                    doc_id=f"doc_{doc_counter}",
                    doc_type=current_doc_type,
                    page_start=current_doc_pages[0].page_num,
                    page_end=current_doc_pages[-1].page_num,
                    text="\n\n".join([p.text for p in current_doc_pages])
                )
                logical_docs.append(logical_doc)
                doc_counter += 1

                # Start new document
                current_doc_type = classify_document_type(page_info.text)
                page_info.doc_type = current_doc_type
                page_info.page_in_doc = 0
                current_doc_pages = [page_info]
                print(f"  Page {i}: New document detected - {current_doc_type}")

    # Don't forget the last document
    if current_doc_pages:
        logical_doc = LogicalDocument(
            doc_id=f"doc_{doc_counter}",
            doc_type=current_doc_type,
            page_start=current_doc_pages[0].page_num,
            page_end=current_doc_pages[-1].page_num,
            text="\n\n".join([p.text for p in current_doc_pages])
        )
        logical_docs.append(logical_doc)

    print(f"Identified {len(logical_docs)} logical documents")
    for ld in logical_docs:
        print(f"   - {ld.doc_type}: Pages {ld.page_start}-{ld.page_end}")

    return pages_info, logical_docs

In [ ]:
# STEP 6: Intelligent Chunking with Metadata Preservation

def chunk_document_with_metadata(logical_doc: LogicalDocument,
                                chunk_size: int = 500,
                                overlap: int = 100) -> List[ChunkMetadata]:
    """
    Chunk a logical document while preserving rich metadata.
    Uses sliding window with overlap for better context.
    """
    chunks_metadata = []
    words = logical_doc.text.split()

    if len(words) <= chunk_size:
        # Document is small enough to be a single chunk
        chunk_meta = ChunkMetadata(
            chunk_id=f"{logical_doc.doc_id}_chunk_0",
            doc_id=logical_doc.doc_id,
            doc_type=logical_doc.doc_type,
            chunk_index=0,
            page_start=logical_doc.page_start,
            page_end=logical_doc.page_end,
            text=logical_doc.text
        )
        chunks_metadata.append(chunk_meta)
    else:
        # Create overlapping chunks
        stride = chunk_size - overlap
        for i, start_idx in enumerate(range(0, len(words), stride)):
            end_idx = min(start_idx + chunk_size, len(words))
            chunk_text = ' '.join(words[start_idx:end_idx])

            # Calculate which pages this chunk spans
            # (simplified - in production, track more precisely)
            chunk_position = start_idx / len(words)
            page_range = logical_doc.page_end - logical_doc.page_start
            relative_page = int(chunk_position * page_range)
            chunk_page_start = logical_doc.page_start + relative_page
            chunk_page_end = min(chunk_page_start + 1, logical_doc.page_end)

            chunk_meta = ChunkMetadata(
                chunk_id=f"{logical_doc.doc_id}_chunk_{i}",
                doc_id=logical_doc.doc_id,
                doc_type=logical_doc.doc_type,
                chunk_index=i,
                page_start=chunk_page_start,
                page_end=chunk_page_end,
                text=chunk_text
            )
            chunks_metadata.append(chunk_meta)

            if end_idx >= len(words):
                break

    return chunks_metadata

def chunk_with_llama_index(logical_doc: LogicalDocument,
                           chunk_size: int = 500,
                           chunk_overlap: int = 100) -> List[Document]:
    """
    Alternative: Use LlamaIndex's advanced chunking with metadata.
    """
    # Create LlamaIndex document with metadata
    doc = Document(
        text=logical_doc.text,
        metadata={
            "doc_id": logical_doc.doc_id,
            "doc_type": logical_doc.doc_type,
            "page_start": logical_doc.page_start,
            "page_end": logical_doc.page_end,
            "source": f"{logical_doc.doc_type}_document"
        }
    )

    # Use LlamaIndex's sentence splitter for better chunking
    splitter = SentenceSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        paragraph_separator="\n\n",
        separator=" ",
    )

    # Create nodes (chunks) from document
    nodes = splitter.get_nodes_from_documents([doc])

    # Convert to our ChunkMetadata format for consistency
    chunks_metadata = []
    for i, node in enumerate(nodes):
        chunk_meta = ChunkMetadata(
            chunk_id=f"{logical_doc.doc_id}_chunk_{i}",
            doc_id=logical_doc.doc_id,
            doc_type=logical_doc.doc_type,
            chunk_index=i,
            page_start=node.metadata.get("page_start", logical_doc.page_start),
            page_end=node.metadata.get("page_end", logical_doc.page_end),
            text=node.text
        )
        chunks_metadata.append(chunk_meta)

    return chunks_metadata

def process_all_documents(logical_docs: List[LogicalDocument],
                         use_llama_index: bool = False) -> List[ChunkMetadata]:
    """
    Process all logical documents into chunks with metadata.
    Can use either custom or LlamaIndex chunking.
    """
    all_chunks = []

    for logical_doc in logical_docs:
        if use_llama_index:
            chunks = chunk_with_llama_index(logical_doc)
        else:
            chunks = chunk_document_with_metadata(logical_doc)

        logical_doc.chunks = chunks  # Store reference
        all_chunks.extend(chunks)
        print(f"  {logical_doc.doc_type}: Created {len(chunks)} chunks")

    return all_chunks

In [ ]:
# STEP 7: Query Routing and Intelligent Retrieval


def predict_query_document_type(query: str) -> Tuple[str, float]:
    """
    Predict which pharmaceutical document type is most likely to contain
    the answer. Returns predicted type and confidence score.
    """
    prompt = f"""Analyze this query and predict which pharmaceutical document type
would most likely contain the answer.

Query: "{query}"

Choose the MOST LIKELY type from:
- Cover Letter: Formal letters about product information or storage conditions
- Certificate Of Quality: Lot numbers, manufacture/expiration dates, test results
- Packaging Specification: Packaging components, materials, part numbers
- Bse/Tse Declaration: Animal-origin material declarations, TSE compliance
- Material Description: Materials of construction, sterilization compatibility
- Supplier Qualification: Supplier audits, ISO certifications, approved products
- Chain Of Custody: Manufactured assemblies, traceability, shipment flow
- Other: General or unclear queries

Respond in JSON format:
{{"type": "DocumentType", "confidence": 0.85}}

Confidence should be between 0.0 and 1.0"""

    try:
        response = llm(prompt, max_tokens=200) # Removed stop and increased max_tokens
        result = json.loads(response['choices'][0]['text'].strip()) # Corrected access
        predicted = result.get("type", "Other")
        confidence = result.get("confidence", 0.5)
        return clean_doc_type(predicted), confidence
    except Exception as e:
        print(f"Query routing error: {e}")
        return "Other", 0.0


class IntelligentRetriever:
    """
    Advanced retrieval system with metadata filtering and query routing.
    """

    def __init__(self):
        self.index = None
        self.chunks_metadata = []
        self.doc_type_indices = {}  # Separate indices per doc type

    def build_indices(self, chunks_metadata: List[ChunkMetadata]):
        """
        Build FAISS indices with document type segregation.
        """
        print("Building vector indices...")
        self.chunks_metadata = chunks_metadata

        # Create embeddings for all chunks
        texts = [chunk.text for chunk in chunks_metadata]
        embeddings = embed_model.encode(texts, show_progress_bar=True)

        # Store embeddings in metadata
        for i, chunk in enumerate(chunks_metadata):
            chunk.embedding = embeddings[i]

        # Build main index
        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dim)
        self.index.add(embeddings)

        # Build separate indices for each document type
        doc_types = set(chunk.doc_type for chunk in chunks_metadata)
        for doc_type in doc_types:
            type_indices = [i for i, chunk in enumerate(chunks_metadata)
                          if chunk.doc_type == doc_type]
            if type_indices:
                type_embeddings = embeddings[type_indices]
                type_index = faiss.IndexFlatL2(dim)
                type_index.add(type_embeddings)
                self.doc_type_indices[doc_type] = {
                    'index': type_index,
                    'mapping': type_indices  # Maps back to original chunks
                }

        print(f"Indexed {len(chunks_metadata)} chunks across {len(doc_types)} document types")

    def retrieve(self, query: str, k: int = 4,
                filter_doc_type: Optional[str] = None,
                auto_route: bool = True) -> List[Tuple[ChunkMetadata, float]]:
        """
        Retrieve relevant chunks with optional filtering and routing.
        Returns chunks with relevance scores.
        """
        query_embedding = embed_model.encode([query])

        # Determine which index to search
        if filter_doc_type and filter_doc_type in self.doc_type_indices:
            # Use filtered index
            type_data = self.doc_type_indices[filter_doc_type]
            D, I = type_data['index'].search(query_embedding, k)
            # Map back to original chunks
            chunk_indices = [type_data['mapping'][i] for i in I[0]]
            distances = D[0]
        elif auto_route:
            # Predict best document type
            predicted_type, confidence = predict_query_document_type(query)
            print(f"Query routed to: {predicted_type} (confidence: {confidence:.2f})")

            if confidence > 0.7 and predicted_type in self.doc_type_indices:
                # High confidence - use specific index
                type_data = self.doc_type_indices[predicted_type]
                D, I = type_data['index'].search(query_embedding, k)
                chunk_indices = [type_data['mapping'][i] for i in I[0]]
                distances = D[0]
            else:
                # Low confidence - search all
                D, I = self.index.search(query_embedding, k)
                chunk_indices = I[0]
                distances = D[0]
        else:
            # Search all chunks
            D, I = self.index.search(query_embedding, k)
            chunk_indices = I[0]
            distances = D[0]

        # Convert distances to similarity scores (inverse)
        max_dist = max(distances) if len(distances) > 0 else 1.0
        scores = [(max_dist - d) / max_dist for d in distances]

        results = [(self.chunks_metadata[i], scores[idx])
                  for idx, i in enumerate(chunk_indices)]

        return results

In [ ]:
# Enhanced answer generation and source attribution
import json

def generate_answer_with_sources(query: str,
                                retrieved_chunks: List[Tuple[ChunkMetadata, float]]) -> Dict:
    """
    Generate answer with detailed source attribution.
    """
    if not retrieved_chunks:
        return {
            'answer': "I couldn't find relevant information to answer your question.",
            'sources': [],
            'confidence': 0.0
        }

    # Prepare context from retrieved chunks
    context_parts = []
    sources = []

    for chunk_meta, score in retrieved_chunks:
        context_parts.append(f"[From {chunk_meta.doc_type}, Pages {chunk_meta.page_start}-{chunk_meta.page_end}]")
        context_parts.append(chunk_meta.text)
        context_parts.append("")

        sources.append({
            'doc_type': chunk_meta.doc_type,
            'pages': f"{chunk_meta.page_start}-{chunk_meta.page_end}",
            'relevance': f"{score:.2%}",
            #'preview': chunk_meta.text  # Store full chunk text here
        })

    context = "\n".join(context_parts)

    # Generate answer
    prompt = f"""You are answering questions about pharmaceutical documentation
including certificates of quality, packaging specifications, and compliance
declarations. Use the provided context to answer the question accurately.
Be specific and cite which document type and pages support your answer.

Context:
{context}

Question: {query}

Instructions:
1. Answer based ONLY on the provided context
2. Mention which document type(s) contain the information
3. Be concise but complete
4. If the context doesn't contain enough information, say so

Answer:"""

    try:
        response = llm(prompt, max_tokens=500) # Added max_tokens for longer answers
        answer = response['choices'][0]['text'].strip() # Corrected access

        # Calculate overall confidence based on retrieval scores
        avg_score = sum(s for _, s in retrieved_chunks) / len(retrieved_chunks)

        return {
            'answer': answer,
            'sources': sources,
            'confidence': avg_score,
            'chunks_used': len(retrieved_chunks)
        }
    except Exception as e:
        print(f"Answer generation error: {e}")
        return {
            'answer': f"Error generating answer: {str(e)}",
            'sources': sources,
            'confidence': 0.0
        }

In [ ]:
# STEP 9: Enhanced Document Store

class EnhancedDocumentStore:
    """
    Manages the complete document processing and retrieval pipeline.
    """

    def __init__(self):
        self.pages_info = []
        self.logical_docs = []
        self.chunks_metadata = []
        self.retriever = IntelligentRetriever()
        self.is_ready = False
        self.processing_stats = {}
        self.filename = None

    def process_pdf(self, pdf_file, filename: str = "document.pdf"):
        """
        Complete PDF processing pipeline.
        """
        self.filename = filename
        self.is_ready = False
        start_time = datetime.now()

        try:
            # Extract and analyze PDF
            self.pages_info, self.logical_docs = extract_and_analyze_pdf(pdf_file)

            # Chunk documents with metadata
            self.chunks_metadata = process_all_documents(self.logical_docs)

            # Build retrieval indices
            self.retriever.build_indices(self.chunks_metadata)

            # Calculate processing statistics
            process_time = (datetime.now() - start_time).total_seconds()
            self.processing_stats = {
                'filename': filename,
                'total_pages': len(self.pages_info),
                'documents_found': len(self.logical_docs),
                'total_chunks': len(self.chunks_metadata),
                'document_types': list(set(doc.doc_type for doc in self.logical_docs)),
                'processing_time': f"{process_time:.1f}s"
            }

            self.is_ready = True
            return True, self.processing_stats

        except Exception as e:
            return False, {'error': str(e)}

    def query(self, question: str, filter_type: Optional[str] = None,
             auto_route: bool = True, k: int = 4) -> Dict:
        """
        Query the document store.
        """
        if not self.is_ready:
            return {
                'answer': "Please upload and process a PDF first.",
                'sources': [],
                'confidence': 0.0
            }

        # Retrieve relevant chunks
        retrieved = self.retriever.retrieve(
            question, k=k,
            filter_doc_type=filter_type,
            auto_route=auto_route
        )

        # Generate answer with sources
        result = generate_answer_with_sources(question, retrieved)
        result['filter_used'] = filter_type or ('auto' if auto_route else 'none')

        return result

    def get_document_structure(self) -> List[Dict]:
        """
        Get the document structure for UI display.
        """
        if not self.logical_docs:
            return []

        structure = []
        for doc in self.logical_docs:
            structure.append({
                'id': doc.doc_id,
                'type': doc.doc_type,
                'pages': f"{doc.page_start + 1}-{doc.page_end + 1}",  # 1-indexed for UI
                'chunks': len(doc.chunks) if doc.chunks else 0,
                'preview': doc.text[:200] + "..." if len(doc.text) > 200 else doc.text
            })

        return structure

In [ ]:
"""
Evaluation harness for the pharma-SDF RAG pipeline (Claude_test.ipynb).

WHAT THIS DOES
---------------
Runs your `doc_store` (an EnhancedDocumentStore instance, already `.process_pdf()`-ed)
against test_questions.json and computes every metric in your
"Pipeline Performance Metrics" template:

  Retrieval:   Recall@K, MRR, Precision@K, Hit Rate
  End-to-end:  Answer Accuracy, Citation Accuracy, Factual Consistency (keyword-based proxy)
  System:      Avg response time, PDF processing time, retrieval latency, generation latency

HOW TO USE (in your Colab notebook, after running all pipeline cells)
-----------------------------------------------------------------------
    import json, time
    with open("test_questions.json") as f:
        test_set = json.load(f)

    # doc_store must already be built:
    #   doc_store = EnhancedDocumentStore()
    #   doc_store.process_pdf(pdf_bytes_or_path, filename="pharma-blob-test.pdf")

    from evaluation_harness import run_evaluation
    results = run_evaluation(doc_store, test_set, k=4)
    print_report(results)

NOTES
-----
- `must_contain` in the test set is a *simple substring proxy* for factual/answer
  correctness. It's a starting point, not a substitute for human review or an
  LLM-judge pass -- swap in your own scorer if you have one.
- Retrieval metrics require your retriever to expose which chunks (and their
  doc_type/page_start) were returned, in ranked order. `run_evaluation` expects
  `doc_store.query()` to return a `sources` list in rank order, matching the
  shape produced in your notebook's `generate_answer_with_sources`.
- This does NOT call any external network -- it only measures your already-running
  pipeline, so it's safe to run inside your existing Colab session.
"""

import time
import statistics
from typing import List, Dict, Any


def _is_relevant(source: Dict, expected_doc_type: str, expected_pages: List[int]) -> bool:
    """A retrieved chunk counts as relevant if its doc_type matches and its
    page range overlaps the expected page range."""
    if source.get("doc_type") != expected_doc_type:
        return False
    pages = source.get("pages", "")
    try:
        lo, hi = (int(p) for p in pages.split("-"))
    except Exception:
        return False
    return not (hi < expected_pages[0] or lo > expected_pages[1])


def run_evaluation(doc_store, test_set: List[Dict[str, Any]], k: int = 4) -> Dict[str, Any]:
    per_query = []

    for item in test_set:
        t0 = time.perf_counter()
        result = doc_store.query(item["query"], k=k)
        t_total = time.perf_counter() - t0

        sources = result.get("sources", [])[:k]
        answer = result.get("answer", "") or ""

        # --- retrieval relevance judgments ---
        relevance = [
            _is_relevant(s, item["expected_doc_type"], item["expected_pages"])
            for s in sources
        ]
        hit = any(relevance)
        first_hit_rank = next((i + 1 for i, r in enumerate(relevance) if r), None)
        num_relevant_retrieved = sum(relevance)

        # --- answer correctness (substring proxy -- replace with your own judge) ---
        answer_lower = answer.lower()
        terms_found = sum(1 for term in item["must_contain"] if term.lower() in answer_lower)
        answer_correct = terms_found == len(item["must_contain"])

        # --- citation accuracy: did the returned sources include the expected doc_type? ---
        citation_correct = any(s.get("doc_type") == item["expected_doc_type"] for s in sources)

        per_query.append({
            "id": item["id"],
            "hit": hit,
            "first_hit_rank": first_hit_rank,
            "precision_at_k": num_relevant_retrieved / max(len(sources), 1),
            "answer_correct": answer_correct,
            "citation_correct": citation_correct,
            "response_time_s": t_total,
        })

    n = len(per_query)
    recall_at_k = sum(1 for q in per_query if q["hit"]) / n          # 1 relevant doc expected per query
    mrr = sum((1 / q["first_hit_rank"]) if q["first_hit_rank"] else 0 for q in per_query) / n
    precision_at_k = sum(q["precision_at_k"] for q in per_query) / n
    hit_rate = recall_at_k  # identical here since each query has exactly one target doc_type/page range
    answer_accuracy = sum(q["answer_correct"] for q in per_query) / n
    citation_accuracy = sum(q["citation_correct"] for q in per_query) / n
    avg_response_time = statistics.mean(q["response_time_s"] for q in per_query)

    return {
        "n_queries": n,
        "recall_at_k": recall_at_k,
        "mrr": mrr,
        "precision_at_k": precision_at_k,
        "hit_rate": hit_rate,
        "answer_accuracy": answer_accuracy,
        "citation_accuracy": citation_accuracy,
        "avg_response_time_s": avg_response_time,
        "per_query": per_query,
    }


def print_report(results: Dict[str, Any]) -> None:
    print(f"Evaluated on {results['n_queries']} test questions\n")
    print("Retrieval Performance")
    print(f"  Recall@K:      {results['recall_at_k']*100:.1f}%")
    print(f"  MRR:           {results['mrr']:.2f}")
    print(f"  Precision@K:   {results['precision_at_k']*100:.1f}%")
    print(f"  Hit Rate:      {results['hit_rate']*100:.1f}%\n")
    print("End-to-End Accuracy")
    print(f"  Answer Accuracy:    {results['answer_accuracy']*100:.1f}%  (n={results['n_queries']})")
    print(f"  Citation Accuracy:  {results['citation_accuracy']*100:.1f}%\n")
    print("System Performance")
    print(f"  Avg Response Time: {results['avg_response_time_s']:.2f}s")


def time_pdf_processing(doc_store, pdf_path: str) -> float:
    """Separately time just the PDF ingestion step (not query time)."""
    t0 = time.perf_counter()
    doc_store.process_pdf(pdf_path)
    return time.perf_counter() - t0


if __name__ == "__main__":
    print(__doc__)

### Download Public Sample PDF for Demonstration

To ensure the evaluation harness can run, a sample PDF document is required. The following cell downloads `pharma-blob-sample.pdf` to the `/content/` directory.

In [ ]:
import os

pdf_file_path = "/content/Another Sample - pharma-blob-test.pdf"

# Check if the PDF already exists to avoid re-downloading
if not os.path.exists(pdf_file_path):
    print(f"Downloading sample PDF to {pdf_file_path}...")
    !wget -q -O {pdf_file_path} https://www.gradio.app/docs/assets/files/pharma-blob-sample.pdf
    print("Download complete.")
else:
    print(f"PDF file already exists at {pdf_file_path}. Skipping download.")


In [ ]:
%%writefile evaluation_harness.py
import time
import statistics
from typing import List, Dict, Any


def _is_relevant(source: Dict, expected_doc_type: str, expected_pages: List[int]) -> bool:
    if source.get("doc_type") != expected_doc_type:
        return False
    pages = source.get("pages", "")
    try:
        lo, hi = (int(p) for p in pages.split("-"))
    except Exception:
        return False
    return not (hi < expected_pages[0] or lo > expected_pages[1])


def run_evaluation(doc_store, test_set: List[Dict[str, Any]], k: int = 4) -> Dict[str, Any]:
    per_query = []

    for item in test_set:
        t0 = time.perf_counter()
        result = doc_store.query(item["query"], k=k)
        t_total = time.perf_counter() - t0

        sources = result.get("sources", [])[:k]
        answer = result.get("answer", "") or ""

        relevance = [
            _is_relevant(s, item["expected_doc_type"], item["expected_pages"])
            for s in sources
        ]
        hit = any(relevance)
        first_hit_rank = next((i + 1 for i, r in enumerate(relevance) if r), None)
        num_relevant_retrieved = sum(relevance)

        answer_lower = answer.lower()
        terms_found = sum(1 for term in item["must_contain"] if term.lower() in answer_lower)
        answer_correct = terms_found == len(item["must_contain"])

        citation_correct = any(s.get("doc_type") == item["expected_doc_type"] for s in sources)

        per_query.append({
            "id": item["id"],
            "hit": hit,
            "first_hit_rank": first_hit_rank,
            "precision_at_k": num_relevant_retrieved / max(len(sources), 1),
            "answer_correct": answer_correct,
            "citation_correct": citation_correct,
            "response_time_s": t_total,
        })

    n = len(per_query)
    recall_at_k = sum(1 for q in per_query if q["hit"]) / n
    mrr = sum((1 / q["first_hit_rank"]) if q["first_hit_rank"] else 0 for q in per_query) / n
    precision_at_k = sum(q["precision_at_k"] for q in per_query) / n
    hit_rate = recall_at_k
    answer_accuracy = sum(q["answer_correct"] for q in per_query) / n
    citation_accuracy = sum(q["citation_correct"] for q in per_query) / n
    avg_response_time = statistics.mean(q["response_time_s"] for q in per_query)

    return {
        "n_queries": n,
        "recall_at_k": recall_at_k,
        "mrr": mrr,
        "precision_at_k": precision_at_k,
        "hit_rate": hit_rate,
        "answer_accuracy": answer_accuracy,
        "citation_accuracy": citation_accuracy,
        "avg_response_time_s": avg_response_time,
        "per_query": per_query,
    }


def print_report(results: Dict[str, Any]) -> None:
    print(f"Evaluated on {results['n_queries']} test questions\n")
    print("Retrieval Performance")
    print(f"  Recall@K:      {results['recall_at_k']*100:.1f}%")
    print(f"  MRR:           {results['mrr']:.2f}")
    print(f"  Precision@K:   {results['precision_at_k']*100:.1f}%")
    print(f"  Hit Rate:      {results['hit_rate']*100:.1f}%\n")
    print("End-to-End Accuracy")
    print(f"  Answer Accuracy:    {results['answer_accuracy']*100:.1f}%  (n={results['n_queries']})")
    print(f"  Citation Accuracy:  {results['citation_accuracy']*100:.1f}%\n")
    print("System Performance")
    print(f"  Avg Response Time: {results['avg_response_time_s']:.2f}s")


def time_pdf_processing(doc_store, pdf_path: str) -> float:
    t0 = time.perf_counter()
    doc_store.process_pdf(pdf_path)
    return time.perf_counter() - t0

In [ ]:
# STEP 10: Gradio Interface

# Global store instance
doc_store = EnhancedDocumentStore()

def process_pdf_handler(pdf_file):
    """Handle PDF upload and processing."""
    if pdf_file is None:
        return "Please upload a PDF file", None, gr.update(choices=["All"])

    # Process the PDF
    success, stats = doc_store.process_pdf(pdf_file,
                                          filename=pdf_file.split('/')[-1] if isinstance(pdf_file, str) else
getattr(pdf_file, 'name', 'pharma-blob-sample.pdf'))

    if success:
        # Prepare status message
        status_msg = f"""
**Successfully Processed:**
- File: {stats['filename']}
- Pages: {stats['total_pages']}
- Documents Found: {stats['documents_found']}
- Chunks Created: {stats['total_chunks']}
- Types: {', '.join(stats['document_types'])}
- Time: {stats['processing_time']}
"""

        # Get document structure for display
        structure = doc_store.get_document_structure()
        structure_display = "\n".join([
            f"- **{doc['type']}** (Pages {doc['pages']}): {doc['chunks']} chunks"
            for doc in structure
        ])

        # Update filter choices
        doc_types = ["All"] + stats['document_types']

        return status_msg, structure_display, gr.update(choices=doc_types, value="All")
    else:
        return f"Error: {stats.get('error', 'Unknown error')}", None, gr.update(choices=["All"])

def chat_handler(message, history, doc_filter, auto_route, num_chunks):
    """Handle chat interactions."""
    if not doc_store.is_ready:
        response = "Please upload and process a pharmaceutical PDF document first."
        return history + [{"role": "user", "content": message}, {"role": "assistant", "content": response}]

    # Query the document store
    filter_type = None if doc_filter == "All" else doc_filter
    result = doc_store.query(
        message,
        filter_type=filter_type,
        auto_route=auto_route and filter_type is None,
        k=num_chunks
    )

    # Format response with sources
    response = f"{result['answer']}\n\n"

    if result['sources']:
        response += "**Sources:**\n"
        for i, src in enumerate(result['sources']):
            response += f"- **Source {i+1}:** {src['doc_type']} (Pages {src['pages']}) - Relevance: {src['relevance']}\n"
            # Removed: response += f"  **Chunk Content:**\n{src['preview']}\n\n" # Display the full chunk content

    response += f"\n*Confidence: {result['confidence']:.1%} | Filter: {result['filter_used']}*"

    return history + [{"role": "user", "content": message}, {"role": "assistant", "content": response}]

def create_interface():
    """Create the Gradio interface for pharmaceutical document Q&A."""

    with gr.Blocks(title="Pharmaceutical Document Q&A System") as demo:
        gr.Markdown("""
        # Pharmaceutical Document Q&A System
        ### Intelligent Multi-Document Analysis with Advanced RAG Pipeline
        Upload a pharmaceutical blob PDF (e.g. pharma-blob-sample.pdf) to identify
        document types, build a searchable index, and ask questions in natural language.
        """)

        with gr.Row():
            # Left side - PDF upload
            with gr.Column(scale=2):
                pdf_input = gr.File(
                    label="Upload Pharmaceutical PDF",
                    file_types=[".pdf"],
                    type="filepath"
                )

                with gr.Row():
                    process_btn = gr.Button(
                        "Process Document",
                        variant="primary",
                        size="lg",
                        scale=2
                    )
                    clear_all_btn = gr.Button(
                        "Clear All",
                        variant="secondary",
                        size="lg",
                        scale=1
                    )

            # Middle - Document info and settings
            with gr.Column(scale=1):
                gr.Markdown("### Document Info")
                status_output = gr.Markdown(
                    value="Waiting for PDF upload..."
                )

                structure_output = gr.Markdown(
                    value="",
                    label="Document Structure"
                )

                gr.Markdown("### Retrieval Settings")

                doc_filter = gr.Dropdown(
                    choices=["All"],
                    value="All",
                    label="Document Type Filter",
                    info="Filter search to a specific pharmaceutical document type"
                )

                auto_route = gr.Checkbox(
                    value=True,
                    label="Auto-Route Queries",
                    info="Automatically detect the most relevant document type"
                )

                num_chunks = gr.Slider(
                    minimum=1,
                    maximum=10,
                    value=2,
                    step=1,
                    label="Chunks to Retrieve"
                )

            # Right side - Chat interface
            with gr.Column(scale=2):
                gr.Markdown("### Ask Questions")
                chatbot = gr.Chatbot(
                    label="Conversation",
                    height=500,
                    elem_id="chatbot",
                    show_label=False,
                )

                with gr.Row():
                    msg_input = gr.Textbox(
                        label="Ask a question",
                        placeholder="e.g., What is the lot number? What sterilization method was used?",
                        scale=4,
                        show_label=False
                    )
                    send_btn = gr.Button("Send", scale=1, variant="primary")

                with gr.Row():
                    clear_chat_btn = gr.Button("Clear Chat", size="sm", scale=1)
                    example_btn1 = gr.Button("Summarise this document", size="sm", scale=1)
                    example_btn2 = gr.Button("Find lot numbers", size="sm", scale=1)

        # Status bar at the bottom
        with gr.Row():
            status_bar = gr.Markdown(
                value="**Status:** Ready | **Documents:** 0 | **Chunks:** 0",
                elem_id="status_bar"
            )

        # Event handlers
        def update_status_bar():
            """Update the status bar with current statistics."""
            if doc_store.is_ready:
                stats = doc_store.processing_stats
                return (
                    f"**Status:** Ready | "
                    f"**Documents:** {stats.get('documents_found', 0)} | "
                    f"**Chunks:** {stats.get('total_chunks', 0)}"
                )
            return "**Status:** Ready | **Documents:** 0 | **Chunks:** 0"

        def clear_all():
            """Clear everything and reset the interface."""
            global doc_store
            doc_store = EnhancedDocumentStore()
            return (
                None,  # pdf_input
                "Waiting for PDF upload...",  # status_output
                "",  # structure_output
                gr.update(choices=["All"], value="All"),  # doc_filter
                [],  # chatbot
                "",  # msg_input
                update_status_bar()  # status_bar
            )

        # Process PDF handler with status bar update
        def process_pdf_with_status(pdf_file):
            status, structure, filter_update = process_pdf_handler(pdf_file)
            status_bar_text = update_status_bar()
            return status, structure, filter_update, status_bar_text

        # Chat handler with status bar update
        def chat_with_status(message, history, doc_filter, auto_route, num_chunks):
            new_history = chat_handler(message, history, doc_filter, auto_route, num_chunks)
            status_bar_text = update_status_bar()
            return new_history, status_bar_text

        # Example question handlers
        def ask_summary(history):
            return chat_handler(
                "Can you provide a summary of the main points in this document?",
                history, doc_filter.value, auto_route.value, num_chunks.value
            )

        def ask_lot_numbers(history):
            return chat_handler(
                "What lot numbers or batch numbers are mentioned in these documents?",
                history, doc_filter.value, auto_route.value, num_chunks.value
            )

        # Wire up all the events
        process_btn.click(
            fn=process_pdf_with_status,
            inputs=[pdf_input],
            outputs=[status_output, structure_output, doc_filter, status_bar]
        )

        clear_all_btn.click(
            fn=clear_all,
            outputs=[pdf_input, status_output, structure_output, doc_filter,
                    chatbot, msg_input, status_bar]
        )

        # Chat interactions
        msg_input.submit(
            fn=chat_with_status,
            inputs=[msg_input, chatbot, doc_filter, auto_route, num_chunks],
            outputs=[chatbot, status_bar]
        ).then(
            lambda: "",
            outputs=[msg_input]
        )

        send_btn.click(
            fn=chat_with_status,
            inputs=[msg_input, chatbot, doc_filter, auto_route, num_chunks],
            outputs=[chatbot, status_bar]
        ).then(
            lambda: "",
            outputs=[msg_input]
        )

        clear_chat_btn.click(
            lambda: [],
            outputs=[chatbot]
        )

        example_btn1.click(
            fn=ask_summary,
            inputs=[chatbot],
            outputs=[chatbot]
        ).then(
            fn=update_status_bar,
            outputs=[status_bar]
        )

        example_btn2.click(
            fn=ask_lot_numbers,
            inputs=[chatbot],
            outputs=[chatbot]
        ).then(
            fn=update_status_bar,
            outputs=[status_bar]
        )

        # Auto-process when PDF is uploaded
        pdf_input.change(
            fn=process_pdf_with_status,
            inputs=[pdf_input],
            outputs=[status_output, structure_output, doc_filter, status_bar]
        )

    return demo

In [ ]:
# STEP 11: Launch the Application

demo = create_interface()
demo.launch(share=True, debug=True, theme=gr.themes.Soft())

## Resources

This RAG pipeline intelligently processes pharmaceutical PDF documents by performing OCR on scanned pages, classifying each logical document within the PDF, and then chunking the content with rich metadata. It leverages a vector index for efficient retrieval, augmented by a query router that predicts the most relevant document type for a given question. Finally, it uses an LLM to generate answers based on the retrieved chunks, complete with source attribution and confidence scores.

## Current Limitations

**1. LLM Stability and Determinism for Classification and Routing**
   - **Specific issue:** The current LLM (`Mistral-7B-Instruct-v0.2`) used for `classify_document_type` and `predict_query_document_type` can sometimes produce inconsistent or non-deterministic results, especially with complex or ambiguous inputs. This leads to misclassification of document types or incorrect query routing, directly impacting retrieval accuracy.
   - **Examples:**
     - A page that is clearly a 'Certificate Of Quality' might occasionally be classified as 'Other' or 'Cover Letter' if the LLM's response deviates from the expected format.
     - A query like "What is the ISO certification for supplier A?" might be routed to 'Material Description' instead of 'Supplier Qualification' if the LLM misinterprets the primary intent.

**2. Limited Context Window for Boundary Detection**
   - **Specific issue:** `detect_document_boundary` relies on sampling the end of the previous page and the start of the current page (500 characters each). If the actual boundary-defining information (e.g., a new document header or a concluding statement) falls outside this small window, the LLM might incorrectly decide that two pages belong to the same document, leading to logical document merges.
   - **Examples:**
     - A new document might start with a blank page or a very short title page, and the significant content indicating a new document only appears much later than the 500-character window.
     - If the concluding remarks of a document are very long, and the new document's identifier is far into the next page, the boundary might be missed.

**3. Suboptimal OCR for Complex Layouts or Low-Quality Scans**
   - **Specific issue:** The reliance on `pytesseract` for OCR on scanned PDFs can be brittle. It struggles with heavily skewed pages, mixed languages, complex table structures, or very low-resolution scans, leading to poor text extraction.
   - **Examples:**
     - Scanned tables might be extracted as a jumbled block of text, making it impossible to correctly interpret tabular data.
     - Watermarks or background noise on scanned documents can introduce numerous recognition errors, leading to gibberish in the extracted text.

**4. Lack of Hybrid Search Capabilities**
   - **Specific issue:** The retriever currently uses pure vector similarity search (dense retrieval). It lacks the ability to combine this with keyword-based search (sparse retrieval) or other methods like BM25.
   - **Why the current approach struggles:** For highly specific queries involving exact terms (e.g., a precise lot number, product code), dense retrieval alone might sometimes miss relevant documents if the semantic meaning is not perfectly captured in the embedding space, or if the relevant chunk is not the top semantic match. Hybrid search often improves recall for such specific queries.

**5. Limited Model Customization/Fine-tuning**
   - **Specific issue:** The current setup uses a pre-trained `Mistral-7B-Instruct-v0.2` and `all-MiniLM-L6-v2` without any fine-tuning on pharmaceutical-specific data.
   - **Why the current approach struggles:** Pharmaceutical documents often contain highly specialized terminology, acronyms, and domain-specific writing styles. General-purpose models may not fully grasp these nuances, potentially leading to less accurate classifications, extractions, or answer generations for highly technical questions.

**6. In-Memory FAISS for Scalability Concerns**
   - **Specific issue:** `FAISS` is an in-memory solution. As the number of processed PDFs and chunks grows into the millions or billions, it will exhaust available RAM.
   - **What would break at larger scale:** The entire RAG pipeline would crash due to out-of-memory errors. It also lacks built-in persistence, meaning the entire index needs to be rebuilt if the application restarts or the data changes, which becomes prohibitively slow for large datasets.

## Proposed Enhancements

### Short-term (Next 2 weeks):

**1. Implement Retry Mechanisms for LLM Calls:**
   - **Specific improvement:** Add retry logic with exponential backoff around all LLM calls (`classify_document_type`, `detect_document_boundary`, `predict_query_document_type`, `generate_answer_with_sources`).
   - **Implementation plan:** Use a library like `tenacity` or custom retry loops with `time.sleep` to re-attempt LLM calls that fail due to temporary API issues or non-deterministic output parsing errors.
   - **Expected impact:** Improves the robustness and reliability of the pipeline by handling transient LLM errors, leading to fewer processing failures and more consistent results.

**2. Enhance Prompt Engineering for LLM Determinism:**
   - **Specific improvement:** Refine prompts for document classification and boundary detection to include more explicit instructions for deterministic output (e.g., "Respond ONLY with 'Yes' or 'No', do not add any other text."), and add more diverse few-shot examples directly in the prompt.
   - **Implementation plan:** A/B test different prompt variations with a small, curated dataset of problematic pages/queries to identify the most robust prompts.
   - **Expected impact:** Reduces LLM's tendency to hallucinate or deviate from the expected output format, improving the accuracy of document type classification and boundary detection.

**3. Improve `detect_document_boundary` with Additional Heuristics:**
   - **Specific improvement:** Supplement the LLM-based boundary detection with additional rule-based heuristics, such as checking for page numbers (e.g., "Page 1 of X" vs. "Page Y of Z"), document IDs, or common new-document keywords at the beginning of pages.
   - **Implementation plan:** Introduce regex patterns or keyword searches in the `detect_document_boundary` function to pre-filter or confirm LLM decisions.
   - **Expected impact:** Increases the accuracy of logical document segmentation, especially for cases where the LLM's context window is insufficient, and reduces false positives/negatives.

### Medium-term (Next month):

**1. Integrate Hybrid Search (BM25 + Vector Search):**
   - **Larger enhancement:** Combine keyword-based search (e.g., using `rank_bm25` or `sparse_retrieval` in `LlamaIndex`) with the existing vector similarity search.
   - **New components/research:** Research and integrate a sparse retriever. Modify the `IntelligentRetriever` to perform both sparse and dense retrieval, and then re-rank the combined results.
   - **Expected impact:** Significantly improves retrieval performance for queries that benefit from exact keyword matches (e.g., product codes, lot numbers) while retaining the semantic understanding of vector search, leading to higher recall and precision.

**2. Advanced OCR Integration (e.g., Google Cloud Vision API):**
   - **Larger enhancement:** Replace `pytesseract` with a more robust, cloud-based OCR solution for handling scanned PDFs.
   - **New components/research:** Integrate with Google Cloud Vision API (or similar commercial OCR service) for superior text extraction quality, especially for challenging documents.
   - **Expected impact:** Drastically improves the quality and completeness of text extraction from low-quality or complex scanned documents, providing better raw input for subsequent RAG pipeline steps and reducing errors downstream.

**3. Implement a Dedicated Evaluation Framework:**
   - **Process improvement:** Develop a comprehensive evaluation dataset (queries + ground truth relevant chunks/documents) and automate the calculation of retrieval metrics (Recall@K, MRR, Precision@K, Hit Rate) and potentially RAG-specific metrics (faithfulness, answer relevance).
   - **Implementation plan:** Create a separate script or notebook to run evaluations on new code changes or prompt adjustments automatically.
   - **Expected impact:** Enables systematic tracking of performance improvements, allows for data-driven decision-making, and ensures that enhancements to one part of the pipeline don't negatively impact others.

### Long-term Vision:

**1. Production-Ready Vector Database Integration:**
   - **Where you'd take this in production:** Migrate from in-memory `FAISS` to a scalable, persistent vector database like **Pinecone, Weaviate, Milvus, or Google Cloud's Vertex AI Vector Search**.
   - **How it could scale/generalize:** This would allow for indexing billions of chunks, provide high availability, robust data management, and native filtering capabilities, essential for a large-scale enterprise solution.
   - **Integration with other systems:** Would involve setting up API keys, client libraries, and possibly ETL pipelines to continuously update the vector store.

**2. Fine-tuning of Embedding and/or LLM Models:**
   - **Where you'd take this in production:** Fine-tune `all-MiniLM-L6-v2` and/or `Mistral-7B-Instruct-v0.2` on a large, domain-specific dataset of pharmaceutical documents.
   - **How it could scale/generalize:** This would significantly enhance the models' understanding of pharmaceutical jargon, improving embedding quality, classification accuracy, and answer generation relevance. Could be generalized to other specialized domains.
   - **Integration with other systems:** Requires a robust MLOps pipeline for data collection, model training, and continuous deployment of fine-tuned models.

**3. User Feedback Loop and Active Learning:**
   - **Where you'd take this in production:** Implement a system where users can provide feedback on the generated answers and source attributions (e.g., "Was this answer helpful?" or "Was this source relevant?").
   - **How it could scale/generalize:** This feedback can be used for active learning, retraining components of the RAG system (e.g., fine-tuning the LLM or re-ranking retrieved documents) to continuously improve performance over time.
   - **Integration with other systems:** Requires building a UI for feedback collection, a data labeling pipeline, and integrating with the model training and deployment infrastructure.

## Component Specifications

| Component            | Technology Choice                                    | Configuration Details                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       |
| :------------------- | :--------------------------------------------------- | :---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| OCR Engine           | `pytesseract` (Python-Tesseract wrapper)             | Used for text extraction on scanned PDF pages when `PyMuPDF` (fitz) fails to extract text.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              |
| Text Chunking        | Custom Sliding Window (default) / `LlamaIndex`'s `SentenceSplitter` | **Custom:** `chunk_size=500` (words), `overlap=100` (words). **LlamaIndex:** `chunk_size=500` (tokens), `chunk_overlap=100` (tokens), `paragraph_separator="\n\n"`, `separator=" "`.                                                                                                                                                                                                                                                                                                                                                                                                                         |
| Embeddings           | `SentenceTransformer`                                | Model: `all-MiniLM-L6-v2`. Used for generating vector embeddings for text chunks and queries.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        |
| Vector Database      | `FAISS` (Facebook AI Similarity Search)              | In-memory `faiss.IndexFlatL2` for L2 distance similarity. Separate indices are built for each document type to support filtered retrieval.                                                                                                                                                                                                                                                                                                                                                                                                                                                               |
| Retriever            | Dense Vector Retrieval with LLM-based Query Routing  | Retrieval method: Similarity search using FAISS. `top-K`: Configurable (default `k=4`). Filtering: Explicit `filter_doc_type` or LLM-driven `auto_route` using `predict_query_document_type`. Confidence threshold of 0.7 for auto-routing.                                                                                                                                                                                                                                                                                                                                                                      |
| LLM                  | `Mistral-7B-Instruct-v0.2`                           | Loaded via `llama_cpp.Llama` with GGUF Q4_K_M quantization. `n_gpu_layers=1` (for GPU acceleration), `n_ctx=2048` (context window size). `max_tokens` varies by task: `predict_query_document_type=200`, `classify_document_type=50`, `detect_document_boundary=10`, `generate_answer_with_sources=500`.                                                                                                                                                                                                                                                                                                         |
| Prompt Strategy      | Instruction-based with Few-Shot Examples (Implicit)  | Detailed, structured prompts are used for document classification, boundary detection, and query routing tasks, often including examples of desired output formats (e.g., JSON). For answer generation, a direct prompt with the query and retrieved context is used, with instructions for summarization and source attribution. Context is directly inserted into the prompt.

## Model Selection Reasoning

### Key Trade-offs Made

#### 🧪 Speed vs. Accuracy:
- **Specific Choices:** We prioritized **local execution speed and resource efficiency** by selecting `Mistral-7B-Instruct-v0.2` and loading it with `llama_cpp.Llama` using **Q4_K_M quantization** and setting `n_gpu_layers=1`. This significantly reduces the model's memory footprint and allows it to run on a Colab's T4 GPU, leading to faster inference times compared to larger, unquantized models. For embeddings, `all-MiniLM-L6-v2` was chosen for its **lightweight nature and speed**. Custom chunking with a sliding window approach was opted for ease of implementation and local context preservation over more complex, potentially slower methods.
- **What you could do differently for better accuracy:** To achieve higher accuracy, one could use a larger, unquantized LLM (e.g., Llama 2 70B, or API-based models like GPT-4 or Gemini Pro). Employing a more powerful or domain-specific embedding model could also improve retrieval relevance. More sophisticated chunking strategies (e.g., semantic chunking, recursive chunking) might enhance the quality of context provided to the LLM.

#### 📍Complexity vs. Maintainability:
- **Explain where you chose simplicity over sophistication:** The use of **FAISS as an in-memory vector database** was a conscious choice for simplicity and ease of setup in a local development environment. It avoids the operational overhead and additional cost associated with managed cloud vector databases. Similarly, the initial **custom sliding window chunking** provided a straightforward way to break down documents.
- **Where you added complexity and why it was worth it:** We introduced **LLM-based intelligent document analysis** (document type classification and boundary detection) during PDF processing. This adds complexity by requiring LLM calls during ingestion but is crucial for segmenting monolithic PDFs into meaningful, logical documents. This pre-processing step vastly improves the quality and relevance of retrieved chunks, making the RAG system more accurate and robust. Additionally, **LLM-based query routing** (predicting the most relevant document type for a query) adds another layer of LLM interaction during retrieval but significantly enhances the system's ability to focus its search on the most promising segments of the document store, improving both precision and recall.

### Decision Rationale / Trade-offs Considered

- **Embedding Model Choice (`SentenceTransformer: all-MiniLM-L6-v2`)**
    - **Rationale:** `all-MiniLM-L6-v2` was selected for its **efficiency, low computational cost, and decent performance** across a wide range of general-purpose semantic similarity tasks. Its small size makes it suitable for local deployment without extensive hardware requirements.
    - **Trade-offs Considered:** The trade-off was between computational efficiency/speed and potentially higher semantic accuracy that might be offered by larger or domain-specific embedding models. For highly specialized pharmaceutical terminology, a more tailored embedding model *could* capture nuances better, but `all-MiniLM-L6-v2` provides a good general baseline.

- **Chunking Strategy (Custom Sliding Window / `LlamaIndex`'s `SentenceSplitter`)**
    - **Rationale:** The **custom sliding window** approach ensures that chunks maintain local context and overlap, which can help in retrieving complete ideas even if they span chunk boundaries. The inclusion of `LlamaIndex`'s `SentenceSplitter` offers an alternative that is more **semantically aware**, splitting text at natural sentence boundaries while respecting `chunk_size` and `chunk_overlap`. This can lead to more coherent chunks that are easier for the LLM to process.
    - **Trade-offs Considered:** Overlapping chunks introduce some redundancy, increasing the total number of chunks and thus the indexing and retrieval time slightly. Finding the optimal `chunk_size` and `overlap` is often an empirical process. Alternatives like fixed-size chunking might be simpler but could break semantic units, while more advanced methods like content-aware chunking or graph-based chunking could offer higher quality but with increased implementation complexity.

- **LLM Choice (`Mistral-7B-Instruct-v0.2` GGUF Q4_K_M)**
    - **Rationale:** `Mistral-7B-Instruct-v0.2` is a strong performer among 7B parameter models, known for its instruction-following capabilities. Running it locally via `llama_cpp.Llama` with **Q4_K_M quantization** directly addresses the need for **cost-effective and private deployment**, avoiding API costs and data privacy concerns. The chosen quantization level (Q4_K_M) strikes a good balance between model size and minimal performance degradation.
    - **Trade-offs Considered:** The primary trade-off is **accuracy and sophistication vs. local deployability and speed**. Larger models or cloud-based proprietary LLMs (e.g., Gemini 1.5 Pro, GPT-4) would likely offer superior reasoning, summarization, and instruction-following abilities. However, these come with higher inference latency, significant costs, and dependence on external APIs. The local Mistral model is a compromise to ensure a self-contained and responsive system on typical hardware.

- **Vector DB Choice (`FAISS: IndexFlatL2`)**
    - **Rationale:** FAISS is selected for its **high performance in similarity search**, making it ideal for the retrieval component. `IndexFlatL2` provides **exact nearest neighbor search**, ensuring high retrieval quality for smaller datasets. The custom segregation of indices by document type, built on top of FAISS, was implemented to support the LLM-based query routing, allowing for highly targeted searches.
    - **Trade-offs Considered:** FAISS is an **in-memory library**, meaning it doesn't offer native persistence, horizontal scalability for massive datasets (beyond what fits in RAM), or advanced filtering capabilities out-of-the-box. For very large-scale deployments or those requiring advanced hybrid search and filtering, a cloud-native vector database (e.g., Pinecone, Weaviate, Milvus, Qdrant) would be more appropriate. These alternatives, however, introduce more complex setup, management, and recurring costs. For this project, the simplicity and performance of in-memory FAISS for a moderate dataset were deemed sufficient.

## RAG Pipeline Workflow

**Document Input** (PDF File) → **Extraction & OCR Processing** (PyMuPDF / pytesseract) → **Intelligent Document Analysis** (LLM for Document Type Classification & Boundary Detection) → **Text Chunking** (Custom Sliding Window or LlamaIndex Sentence Splitter) → **Embedding Generation** (SentenceTransformer) → **Vector Indexing & Storage** (FAISS with segregated indices by Document Type) → **Query Routing** (LLM-based Document Type Prediction) → **Contextual Retrieval** (FAISS search with filters) → **LLM Answer Generation** (Mistral-7B-Instruct-v0.2) → **Attributed Output** (Answer with Sources and Confidence).

## 🧪 Retrieval Performance Metrics

To measure retrieval performance, you'll need a dataset of queries and their associated relevant document IDs (ground truth). The following functions provide implementations for common Information Retrieval (IR) metrics like Recall@K, Mean Reciprocal Rank (MRR), Precision@K, and Hit Rate.

In [ ]:
import numpy as np

def calculate_recall_at_k(retrieved_ids: list, relevant_ids: list, k: int) -> float:
    """
    Calculates Recall@K.
    retrieved_ids: List of document IDs retrieved for a query (ordered by relevance).
    relevant_ids: List of ground truth relevant document IDs for the query.
    k: The 'k' in Recall@K.
    """
    retrieved_at_k = set(retrieved_ids[:k])
    relevant_set = set(relevant_ids)

    if not relevant_set:
        return 1.0  # No relevant documents, perfect recall if nothing is missed

    # Number of relevant documents retrieved within the top K
    num_relevant_retrieved = len(retrieved_at_k.intersection(relevant_set))

    # Total number of relevant documents
    total_relevant = len(relevant_set)

    return num_relevant_retrieved / total_relevant

def calculate_mrr(retrieved_ids: list, relevant_ids: list) -> float:
    """
    Calculates Mean Reciprocal Rank (MRR) for a single query.
    retrieved_ids: List of document IDs retrieved for a query (ordered by relevance).
    relevant_ids: List of ground truth relevant document IDs for the query.
    """
    for i, doc_id in enumerate(retrieved_ids):
        if doc_id in relevant_ids:
            return 1.0 / (i + 1)  # Reciprocal rank for the first relevant document
    return 0.0  # No relevant documents found

def calculate_precision_at_k(retrieved_ids: list, relevant_ids: list, k: int) -> float:
    """
    Calculates Precision@K.
    retrieved_ids: List of document IDs retrieved for a query (ordered by relevance).
    relevant_ids: List of ground truth relevant document IDs for the query.
    k: The 'k' in Precision@K.
    """
    retrieved_at_k = set(retrieved_ids[:k])
    relevant_set = set(relevant_ids)

    if not retrieved_at_k:
        return 0.0 # No documents retrieved, precision is 0

    # Number of relevant documents retrieved within the top K
    num_relevant_retrieved = len(retrieved_at_k.intersection(relevant_set))

    return num_relevant_retrieved / k

def calculate_hit_rate(retrieved_ids: list, relevant_ids: list) -> int:
    """
    Calculates Hit Rate for a single query (1 if at least one relevant doc is retrieved, 0 otherwise).
    retrieved_ids: List of document IDs retrieved for a query (ordered by relevance).
    relevant_ids: List of ground truth relevant document IDs for the query.
    """
    relevant_set = set(relevant_ids)
    for doc_id in retrieved_ids:
        if doc_id in relevant_set:
            return 1
    return 0




### Example Usage and Aggregation

To use these metrics, you would typically run your RAG pipeline's retrieval step for a set of test queries, collect the retrieved document IDs, and compare them against your pre-defined relevant document IDs (ground truth).

Below is a conceptual example. You will need to replace the `test_queries` and `ground_truth_relevant_docs` with your actual evaluation dataset.

In [ ]:
if doc_store.is_ready:
    document_structure = doc_store.get_document_structure()
    print("\n--- Document Structure ---")
    for doc_summary in document_structure:
        print(f"ID: {doc_summary['id']}, Type: {doc_summary['type']}, Pages: {doc_summary['pages']}, Preview: {doc_summary['preview'][:100]}...")
else:
    print("Please process a PDF first to view the document structure.")